# AMES Assignment 1 — Salary Data Analysis   - Naveen Kumar N
**Analytical Methods in Engineering Systems**

Dataset: `Salary_dataset.csv` — Years of Experience vs. Annual Salary of engineers.

This notebook walks through data typing, sampling, descriptive statistics, visualization,
probability, distributions, the Central Limit Theorem, confidence intervals, hypothesis
testing (parametric and non-parametric), ANOVA, and a concluding management report.

*Every code cell is commented line-by-line to explain what it does and why.*

In [6]:
import numpy as np                              # numerical computing: arrays, random numbers, math functions
import pandas as pd                             # data handling: DataFrames, CSV reading, groupby, etc.
import matplotlib.pyplot as plt                 # low-level plotting library
import seaborn as sns                           # nicer statistical plotting, built on matplotlib
from scipy import stats                         # statistical tests, distributions (t, normal, poisson, etc.)
import statsmodels.api as sm                     # regression/ANOVA models and formal statistical tables
from statsmodels.formula.api import ols          # lets us write ANOVA models using an R-style formula string

np.random.seed(42)                              # fixes NumPy's global random seed so old-style random calls are reproducible
sns.set_style("whitegrid")                      # sets a clean seaborn theme (white background + grid lines) for all plots
plt.rcParams['figure.figsize'] = (7, 4)         # sets a sensible default figure size for every plot in this notebook

pd.set_option('display.precision', 2)           # display floats in DataFrames with 2 decimal places, purely cosmetic
BOLD = "\033[1m"                                # A cleaner approach to start bold in output
RESET = "\033[0m"                               # A cleaner approach to end bold in output

## Task 1: Understanding Data Types & Basic Structure

### 1.1 Load the dataset

In [7]:
df = pd.read_csv("C:\\Users\\LENOVO\\anaconda3\\study\\AMES\\Assignment-Lab1\\Salary_dataset.csv", index_col=0) 
                                                       # read the CSV into a DataFrame; column 0 (the unnamed index) becomes the row index
print("Shape:", df.shape)                              # (rows, columns) -> confirms how many engineers/records we have
df.head(5)                                            # show the rows so we can visually sanity-check the data

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\LENOVO\\anaconda3\\study\\AMES\\Assignment-Lab1\\Salary_dataset.csv'

In [ ]:
print("************************************************")
df.info()                                              # column names, non-null counts, and dtypes -> checks for missing values/wrong types 
print("************************************************")
df.describe()                                          # quick summary stats (count, mean, std, min, quartiles, max) for numeric columns 

### 1.2(a) Variable types

| Variable | Qualitative/Quantitative | Discrete/Continuous | Level of Measurement |
|---|---|---|---|
| `YearsExperience` | Quantitative | Continuous (measured, can take any real value, e.g. 4.2 years) | **Ratio** — has a true zero (0 years experience) and equal, meaningful ratios (10 yrs is twice 5 yrs) |
| `Salary` | Quantitative | Continuous | **Ratio** — true zero (₹0 salary) and ratios are meaningful (a ₹100,000 salary is twice a ₹50,000 salary) |

Both variables are quantitative-continuous-ratio scale, which is why arithmetic operations
(mean, ratios, coefficient of variation) are all valid on them.

### 1.2(b) Population vs. Sample

- **Population (conceptual):** All engineers currently employed by the company (or, more broadly,
  all engineers in the relevant labor market whose experience–salary relationship the company
  wants to understand). In this assignment, for the purpose of the sampling exercises in Task 2,
  we **treat the entire 30-row dataset as the population** of engineers in the company.
- **Sample:** Any subset of engineers actually observed/selected for analysis — e.g., the
  simple random, systematic, or stratified samples of size 10/20 drawn in Task 2, or a survey
  of a subset of employees rather than all of them.

## Task 2: Sampling & Sampling Methods

We treat the full dataset (n = 30) as the **population**. We now draw samples of size
n = 10 and n = 20 using three methods.

In [ ]:
N = len(df)                                    # population size = total number of rows (engineers) in the dataset
population_mean = df['Salary'].mean()          # arithmetic mean of Salary across the ENTIRE population
population_std = df['Salary'].std(ddof=0)      # population standard deviation; ddof=0 divides by N (not N-1) since this is the population
print(f"Population size N = {N}")
print(f"Population mean salary = {population_mean:,.2f}")
print(f"Population std salary (ddof=0) = {population_std:,.2f}")

### 2.1(a) Simple Random Sampling (SRS)

In [ ]:
def simple_random_sample(data, n, seed=1):
    return data.sample(n=n, random_state=seed)  # pandas picks n rows uniformly at random, no replacement; random_state fixes the seed for reproducibility

srs_10 = simple_random_sample(df, 10, seed=1)   # draw a random sample of 10 engineers
srs_20 = simple_random_sample(df, 20, seed=1)   # draw a random sample of 20 engineers

print("**************  Simple Random Sample1 -> srs_10 -> n=10  *********************")
print(srs_10)
print("------------------------------------------------------")      
print("Sample 1 - SRS (n=10): mean = ", round(srs_10['Salary'].mean(),2),   # sample mean salary for the n=10 SRS sample
      " std =", round(srs_10['Salary'].std(),2))                           # sample std dev of salary for the n=10 SRS sample 
print("-------------------------------------------------------") 
print(" ")
print("**************  Simple Random Sample2 -> srs_20 -> n=20  *********************")
print(srs_20)
print("-------------------------------------------------------")      
print("Sample 2 - SRS (n=20): mean =", round(srs_20['Salary'].mean(),2),   # sample mean salary for the n=20 SRS sample
      " std =", round(srs_20['Salary'].std(),2))                           # sample std dev of salary for the n=20 SRS sample
print("*******************************************************")

*Explanation:* `DataFrame.sample()` gives every row an equal probability of selection with
no replacement — this is a direct implementation of simple random sampling.

### 2.1(b) Systematic Sampling (every k-th record after a random start)

In [ ]:
def systematic_sample(data, n, seed=1):
    data = data.reset_index(drop=True)          # renumber rows 0..N-1 so we can step through them with a fixed interval k
    N = len(data)                               # population size after reset
    k = N // n                                  # sampling interval: how many records to skip between picks (integer division)
    rng = np.random.default_rng(seed)           # create a seeded random number generator (modern NumPy API)
    start = rng.integers(0, k)                  # pick a random starting point between 0 and k-1 (inclusive of 0, exclusive of k)
    idx = np.arange(start, N, k)[:n]            # build indices start, start+k, start+2k, ... and keep only the first n of them
    return data.loc[idx], k, start              # return the selected rows plus k and start (for reporting)

sys_10, k10, start10 = systematic_sample(df, 10, seed=1)   # systematic sample of size 10
sys_20, k20, start20 = systematic_sample(df, 20, seed=1)   # systematic sample of size 20

print("**************  Systematic Sampling Sample 1 -> sample size = 10 after every Kth record  **********\n")
print(f" n=10 -> interval k={k10}, random start={start10}")
print("Systematic (n=10): mean =", round(sys_10['Salary'].mean(),2),
      " std =", round(sys_10['Salary'].std(),2))
print("\n----------------Details of the sample 1 for above------------")
print(sys_10)


print(" ") 
print("**************  Systematic Sampling Sample 2 -> sample size = 20 after every Kth record  **********\n")
print(f"n=20 -> interval k={k20}, random start={start20}")
print("Systematic (n=20): mean =", round(sys_20['Salary'].mean(),2),
      " std =", round(sys_20['Salary'].std(),2)) 
print("\n----------------Details of the sample 2 for above------------")
print(sys_20) 


*Explanation:* the sampling interval is k = N/n. A random starting point between 0 and
k-1 is chosen, then every k-th record from that start is selected. This spreads the sample
evenly across the (index-)ordered population.

### 2.1(c) Stratified Sampling (Junior / Mid / Senior)

In [ ]:
def make_strata(data):
    conditions = [
        data['YearsExperience'] < 4,                                    # condition 1: Junior
        (data['YearsExperience'] >= 4) & (data['YearsExperience'] < 8), # condition 2: Mid
        data['YearsExperience'] >= 8                                    # condition 3: Senior
    ]
    choices = ['Junior', 'Mid', 'Senior']            # label to assign for each corresponding condition above
    data = data.copy()                               # copy the DataFrame so we don't accidentally modify the original df
    data['Stratum'] = np.select(conditions, choices, default='Unknown')  # vectorized if/elif/else: assign a label per row based on which condition is True
    return data

df_strat = make_strata(df)                           # apply the stratification to the full dataset
print("-------First building Stratum for all the data frame---------")  
print(df_strat['Stratum'].value_counts())            # count how many engineers fall into each stratum
print("--------Verifying data frame output-----------------") 
print(df_strat)

def stratified_sample(data, n, seed=1):
    # proportional allocation across strata, at least 1 per stratum
    data = make_strata(data)                                     # add the Stratum column
    props = data['Stratum'].value_counts(normalize=True)         # proportion of the population in each stratum (sums to 1)
    alloc = (props * n).round().astype(int)                      # how many sample units each stratum should get, proportional to its size
    alloc[alloc == 0] = 1                                        # guarantee every stratum gets at least 1 unit, even if its share rounds to 0
    # adjust rounding so total == n
    while alloc.sum() > n:                                       # if rounding gave us too many total units...
        alloc[alloc.idxmax()] -= 1                               # ...remove one from the currently largest-allocated stratum
    while alloc.sum() < n:                                       # if rounding gave us too few total units...
        alloc[alloc.idxmin()] += 1                               # ...add one to the currently smallest-allocated stratum
    parts = [data[data['Stratum']==s].sample(n=min(cnt, (data['Stratum']==s).sum()),  # sample 'cnt' rows from stratum s (capped at stratum size)
                                              random_state=seed)
             for s, cnt in alloc.items()]                        # repeat for every (stratum, count) pair
    return pd.concat(parts)                                      # stack all the per-stratum samples into one DataFrame

strat_10 = stratified_sample(df, 10, seed=1)         # stratified sample of size 10
strat_20 = stratified_sample(df, 20, seed=1)         # stratified sample of size 20

print(" ")
print("********************* For Sample 1 -> n=10 ********************************")
print("\nStratified (n=10): mean =", round(strat_10['Salary'].mean(),2),
      " std =", round(strat_10['Salary'].std(),2))
print(strat_10['Stratum'].value_counts() if 'Stratum' in strat_10 else "") 
print("\n---------stratified sample of size 10 for validation of output  ----------\n")
print(strat_10)
print(" ")
print("********************* For Sample 2 -> n=20 ********************************")
print("Stratified (n=20): mean =", round(strat_20['Salary'].mean(),2),
      " std =", round(strat_20['Salary'].std(),2)) 
print(strat_20['Stratum'].value_counts() if 'Stratum' in strat_20 else "") 
print("\n---------stratified sample of size 20 for validation of output  ----------\n")
print(strat_20)

*Explanation:* engineers are split into three experience-based strata (Junior <4 yrs,
Mid 4–8 yrs, Senior ≥8 yrs). Sample size is allocated proportionally to each stratum's share
of the population, then SRS is applied **within** each stratum. This guarantees representation
of all experience levels, unlike plain SRS which could by chance miss a group.

### 2.2 Comparison of sampling methods against the population

In [ ]:
summary = pd.DataFrame({
    'Method': ['Population', 'SRS (n=10)', 'SRS (n=20)',                  # row labels for the comparison table
               'Systematic (n=10)', 'Systematic (n=20)',
               'Stratified (n=10)', 'Stratified (n=20)'],
    'Mean Salary': [population_mean, srs_10['Salary'].mean(), srs_20['Salary'].mean(),   # mean salary for each method, in the same order as 'Method'
                     sys_10['Salary'].mean(), sys_20['Salary'].mean(),
                     strat_10['Salary'].mean(), strat_20['Salary'].mean()],
    'Std Salary': [population_std, srs_10['Salary'].std(), srs_20['Salary'].std(),        # std dev of salary for each method
                   sys_10['Salary'].std(), sys_20['Salary'].std(),
                   strat_10['Salary'].std(), strat_20['Salary'].std()],
})
summary['Abs Mean Diff from Pop'] = (summary['Mean Salary'] - population_mean).abs()  # new column: how far each method's mean is from the true population mean
summary                                                                                # display the table (last line of a cell auto-displays in Jupyter)

**Comment:** Stratified sampling generally produces sample means closest to the
population mean, especially as it explicitly preserves the proportion of Junior/Mid/Senior
engineers — the main source of salary variation in this dataset. Since salary is strongly
driven by experience level, forcing representation across experience strata reduces
sampling error relative to SRS or systematic sampling, which can by chance over- or
under-represent one experience band (particularly at n = 10). Larger sample sizes (n = 20)
bring all three methods closer to the population values, consistent with the law of large
numbers — sampling error shrinks as n increases regardless of method.

### 2.3 Parameter vs. Statistic

- **Parameter** — a numerical summary of the **population** (usually unknown, fixed).
  Examples in this context:
  1. The true mean salary of *all* engineers in the company, μ.
  2. The true standard deviation of salary across *all* engineers, σ.
- **Statistic** — a numerical summary computed from a **sample**, used to estimate a
  parameter (varies from sample to sample). Examples in this context:
  1. The sample mean salary from the n = 20 stratified sample, x̄.
  2. The sample standard deviation of salary from the n = 10 SRS sample, s.

## Task 3: Descriptive Statistics & Point Estimation

In [ ]:
sample20 = strat_20.copy()   # chosen sample of size 20 from Task 2 (using the stratified sample); .copy() avoids modifying strat_20 later

def describe_var(series, name):
    out = {
        'Mean': series.mean(),                                     # arithmetic average
        'Median': series.median(),                                 # middle value when sorted
        'Mode': series.mode().iloc[0],                              # most frequent value (first one if there's a tie)
        'Range': series.max() - series.min(),                       # spread from smallest to largest value
        'Variance': series.var(),                                   # average squared deviation from the mean (sample variance, ddof=1 default)
        'Std Dev': series.std(),                                    # square root of variance, same units as the data
        'IQR': series.quantile(0.75) - series.quantile(0.25),       # interquartile range: spread of the middle 50% of data
        'CoV (%)': (series.std() / series.mean()) * 100             # coefficient of variation: std expressed as a % of the mean, for comparing spread across variables
    }
    return pd.Series(out, name=name)                                # package the dict into a labeled Series (one column of our comparison table)

desc_table = pd.concat([
    describe_var(df['Salary'], 'Salary (Full data)'),               # stats for Salary using the whole population
    describe_var(sample20['Salary'], 'Salary (Sample n=20)'),       # stats for Salary using only the n=20 sample
    describe_var(df['YearsExperience'], 'YearsExperience (Full data)'),        # stats for YearsExperience, full data
    describe_var(sample20['YearsExperience'], 'YearsExperience (Sample n=20)'), # stats for YearsExperience, sample
], axis=1)                                                            # axis=1 places each Series as a column side by side
desc_table

**Inference:**
- Salary shows a Coefficient of Variation (CoV) around 25–30%, meaning salary dispersion is
  moderate relative to its mean — expected, since salary rises fairly steadily with experience
  rather than being erratic.
- YearsExperience has a *higher* CoV than Salary in relative terms, since it is spread from
  ~1 to ~10 years with a mean around 5 — small absolute spread but proportionally large
  relative to its own mean.
- The n = 20 stratified sample's descriptive statistics track the full-data statistics closely
  (mean and spread are similar), which supports the sample being a reasonable point-estimate
  source for the population.

### Point Estimate

The **sample mean salary from the n = 20 sample** (x̄ ≈ value above) is a *point estimate*
of the population mean salary μ.

- **Why "point estimate":** it is a single numerical value (a "point") computed from sample
  data that serves as the best guess for the unknown population parameter — as opposed to an
  *interval estimate*, which gives a range of plausible values.
- **How it might change across samples:** since it depends on which 20 engineers happen to be
  drawn, a different random sample of size 20 will almost certainly give a slightly different
  x̄. This sample-to-sample variability is exactly what the *sampling distribution* (Task 7)
  and *confidence intervals* (Task 8) formally quantify.
- **Engineering/HR interpretation:**
  - The **mean salary** tells management the "typical" pay level of an engineer in the company —
    useful as a single benchmark figure for budgeting and for comparison against industry
    averages.
  - A **high standard deviation** of salary indicates wide pay dispersion — could reflect a
    healthy experience-based pay ladder, but also raises internal-equity or pay-compression
    risk if not well explained by legitimate factors (experience, role, performance). A
    **low standard deviation** suggests salaries are compressed/uniform — which may be fair but
    could also fail to reward experience or performance adequately.

## Task 4: Data Visualization, Correlation & Covariance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))          # create a 2x2 grid of subplots; axes is a 2x2 array of subplot objects

axes[0,0].hist(df['Salary'], bins=8, color='steelblue', edgecolor='black')   # top-left: histogram of Salary, 8 bins
axes[0,0].set_title('Histogram of Salary')
axes[0,0].set_xlabel('Salary'); axes[0,0].set_ylabel('Frequency')

axes[0,1].hist(df['YearsExperience'], bins=8, color='seagreen', edgecolor='black')  # top-right: histogram of YearsExperience
axes[0,1].set_title('Histogram of YearsExperience')
axes[0,1].set_xlabel('Years of Experience'); axes[0,1].set_ylabel('Frequency')

axes[1,0].boxplot(df['Salary'])                # bottom-left: vertical boxplot of Salary (shows median, quartiles, outliers)
axes[1,0].set_title('Boxplot of Salary')
axes[1,0].set_ylabel('Salary')

axes[1,1].scatter(df['YearsExperience'], df['Salary'], color='darkorange', edgecolor='black')  # bottom-right: scatter of Salary vs YearsExperience
axes[1,1].set_title('Salary vs YearsExperience')
axes[1,1].set_xlabel('Years of Experience'); axes[1,1].set_ylabel('Salary')

plt.tight_layout()      # auto-adjust spacing so titles/labels don't overlap between subplots
plt.show()               # render the figure

### 4.2 Visual comment on the scatter plot
The scatter plot shows a clear **positive** relationship between years of experience and
salary — as experience increases, salary increases. The pattern looks **fairly linear**,
with points lying close to an upward-sloping straight line and no strong curvature or
obvious outliers, suggesting a simple linear model would fit reasonably well.

In [ ]:
covariance = df['YearsExperience'].cov(df['Salary'])     # covariance: measures how the two variables vary together (sign shows direction, magnitude depends on units)
correlation = df['YearsExperience'].corr(df['Salary'])   # Pearson correlation: covariance standardized to always fall between -1 and +1
print(f"Covariance(YearsExperience, Salary) = {covariance:,.2f}")
print(f"Pearson correlation coefficient r = {correlation:.4f}")

**Interpretation:** The correlation coefficient is strongly **positive** (close to +1),
confirming that experience and salary move together closely and the relationship is close to
linear. In HR/policy terms, this supports an experience-based compensation structure — the
company's salary policy appears to reward tenure/experience consistently, which is generally
seen as fair and predictable, though it also means pay is largely explained by a single factor
(experience), and other legitimate drivers (skills, performance, role) may be under-weighted.

## Task 5: Probability Concepts with the Salary Data

**Sample space Ω (Omega)(in words):** the set of all 30 engineers in the dataset — i.e., every
possible outcome of "randomly picking one engineer from the company" is one engineer in Ω.

**Events:**
- **A:** "Engineer has more than 5 years of experience" (`YearsExperience > 5`)
- **B:** "Engineer's salary is above 90,000" (`Salary > 90000`)

In [ ]:
A = df['YearsExperience'] > 5        # a Boolean Series: True for every engineer who satisfies event A
B = df['Salary'] > 90000             # a Boolean Series: True for every engineer who satisfies event B

P_A = A.mean()                       # mean of a Boolean series = proportion of True values = empirical P(A)
P_B = B.mean()                       # empirical P(B)
P_A_and_B = (A & B).mean()           # element-wise AND, then take the mean -> empirical P(A ∩ B)
P_A_or_B = (A | B).mean()            # element-wise OR, then take the mean -> empirical P(A ∪ B)

print(f"P(A) = {P_A:.3f}")
print(f"P(B) = {P_B:.3f}")
print(f"P(A ∩ B) = {P_A_and_B:.3f}")
print(f"P(A ∪ B) = {P_A_or_B:.3f}")

P_A_given_B = P_A_and_B / P_B if P_B > 0 else np.nan   # conditional probability formula: P(A|B) = P(A∩B)/P(B); guard against division by zero
P_B_given_A = P_A_and_B / P_A if P_A > 0 else np.nan   # P(B|A) = P(A∩B)/P(A)
print(f"P(A|B) = {P_A_given_B:.3f}")
print(f"P(B|A) = {P_B_given_A:.3f}") 

#print("****************** \n YearsExperience > 5 :: \n", A ) 
#print("****************** \n Salary > 90000 :: \n", B ) 


In [ ]:
# Check probability axioms
print("P(Omega) = 1.0 (always true, whole sample space)")             # by definition, the probability of the entire sample space is 1
print("0 <= P(A) <= 1 :", 0 <= P_A <= 1)                              # axiom check: every probability must lie in [0,1]
print("0 <= P(B) <= 1 :", 0 <= P_B <= 1)
addition_rule_check = P_A + P_B - P_A_and_B                            # the inclusion-exclusion formula for P(A∪B)
print(f"P(A) + P(B) - P(A∩B) = {addition_rule_check:.3f}  vs directly computed P(A∪B) = {P_A_or_B:.3f}")   # compare formula result to the directly computed union probability

# Independence check: P(A ∩ B) vs P(A)*P(B)
print(f"\nP(A)*P(B) = {P_A*P_B:.3f}  vs  P(A ∩ B) = {P_A_and_B:.3f}")  # if these two numbers are equal, A and B are independent

**Comment on independence:** For two events to be independent, P(A ∩ B) must equal
P(A) × P(B) (equivalently P(A|B) = P(A)). Here P(A ∩ B) is noticeably **larger** than
P(A)×P(B), and P(A|B) is much higher than the unconditional P(A) — i.e., knowing an engineer
earns above 90,000 makes it *more* likely they have over 5 years of experience. So **A and B
are not independent**; they are positively associated, which is consistent with the strong
positive correlation between experience and salary found in Task 4.

**Axiom check:** P(Ω) = 1 by definition; both P(A) and P(B) lie in [0, 1]; and the addition
rule P(A∪B) = P(A) + P(B) − P(A∩B) holds exactly (up to rounding) as shown above — consistent
with the Kolmogorov probability axioms.

## Task 6: Random Variables, Distributions & Expected Values

### 6.1 Bernoulli random variable X

In [ ]:
df['X'] = (df['YearsExperience'] >= 5).astype(int)   # new column: 1 if experience >= 5 years, else 0 -> this is our Bernoulli random variable
p_hat = df['X'].mean()                                # sample proportion of 1's = estimate of the Bernoulli parameter p

E_X_theory = p_hat                                    # Bernoulli formula: E[X] = p
Var_X_theory = p_hat * (1 - p_hat)                    # Bernoulli formula: Var(X) = p(1-p)

E_X_empirical = df['X'].mean()                        # empirical mean of the actual X column (same as p_hat here)
Var_X_empirical = df['X'].var(ddof=0)                 # empirical variance of X; ddof=0 because we're treating X's values as the full population of interest

print(f"Estimated p = {p_hat:.3f}")
print(f"Theoretical  E[X] = p = {E_X_theory:.3f},  Var(X) = p(1-p) = {Var_X_theory:.3f}")
print(f"Empirical    E[X] = {E_X_empirical:.3f},  Var(X) = {Var_X_empirical:.3f}")

X is Bernoulli(p) because it takes only two values (1 = "≥5 yrs experience", 0 = otherwise),
with P(X=1) = p estimated directly as the sample proportion. The theoretical Bernoulli formulas
E[X] = p and Var(X) = p(1−p) match the empirical mean/variance of X exactly (by construction,
since p̂ is estimated from the same data) — confirming the Bernoulli model is an appropriate,
internally consistent description of X.

### 6.2 Binomial random variable Y

In [ ]:
n_sample = 15                                          # hypothetical sample size for Y (number of engineers we'd look at)
p_hat_Y = (df['Salary'] > 100000).mean()               # estimate p = probability a single engineer earns > 100,000, from the data
print(f"Approx. probability an engineer earns > 100,000: p ≈ {p_hat_Y:.3f}")

E_Y = n_sample * p_hat_Y                               # Binomial formula: E[Y] = n*p
Var_Y = n_sample * p_hat_Y * (1 - p_hat_Y)             # Binomial formula: Var(Y) = n*p*(1-p)
print(f"For a sample of n = {n_sample} engineers:")
print(f"  Theoretical E[Y] = n*p = {E_Y:.3f}")
print(f"  Theoretical Var(Y) = n*p*(1-p) = {Var_Y:.3f}")

**Conceptual argument:** Y = "number of engineers (out of a sample of n) whose Salary >
100,000" can be modeled as **Binomial(n, p)** because it counts the number of "successes"
(Salary > 100,000) across n roughly independent Bernoulli trials (each engineer either does or
doesn't exceed the threshold), each with the same success probability p estimated from the
population data.

### 6.3 Poisson model for hiring

In [ ]:
lam = 5  # hypothetical mean number of engineers hired per month (the Poisson rate parameter, lambda)

P_N0 = stats.poisson.pmf(0, lam)               # P(N=0): probability mass function of Poisson(lambda) evaluated at 0
P_N1 = stats.poisson.pmf(1, lam)               # P(N=1)
P_N_ge_5 = 1 - stats.poisson.cdf(4, lam)       # P(N>=5) = 1 - P(N<=4); cdf(4) gives P(N<=4) (cumulative distribution function)

print(f"Poisson(λ=5): P(N=0) = {P_N0:.4f}")
print(f"Poisson(λ=5): P(N=1) = {P_N1:.4f}")
print(f"Poisson(λ=5): P(N>=5) = {P_N_ge_5:.4f}")

**HR planning use:** a Poisson(λ=5) hiring model lets HR estimate, e.g., the probability of
zero hires in a slow month (useful for workforce-planning contingency) or of an unusually busy
month (≥5 hires, useful for onboarding/capacity planning), helping size recruitment resources
and budget month-to-month rather than relying on a single average figure.

### 6.4 Normal distribution fit to Salary

In [ ]:
mu = df['Salary'].mean()      # fit a Normal distribution to Salary by using the sample mean as mu...
sigma = df['Salary'].std()    # ...and the sample standard deviation as sigma (method-of-moments fit)
print(f"Fitted Normal: mu = {mu:,.2f}, sigma = {sigma:,.2f}")

# P(60,000 < Salary < 100,000)
p_range = stats.norm.cdf(100000, mu, sigma) - stats.norm.cdf(60000, mu, sigma)  # P(X<100000) - P(X<60000) = P(60000<X<100000), using the Normal CDF
print(f"P(60,000 < Salary < 100,000) ≈ {p_range:.4f}")

# 90th percentile
pct90_theoretical = stats.norm.ppf(0.90, mu, sigma)   # ppf = inverse CDF; gives the value below which 90% of the fitted Normal distribution lies
pct90_empirical = np.percentile(df['Salary'], 90)     # the actual (empirical) 90th percentile computed directly from the raw Salary data
print(f"90th percentile (Normal model)  ≈ {pct90_theoretical:,.2f}")
print(f"90th percentile (empirical, np.percentile) = {pct90_empirical:,.2f}")

## Task 7: Sampling Distribution & Central Limit Theorem (CLT)

In [ ]:
def simulate_sampling_dist(data, n, reps=1000, seed=42):
    rng = np.random.default_rng(seed)              # seeded random generator, so results are reproducible
    means = []                                     # empty list to collect the mean of each simulated sample
    values = data['Salary'].values                 # pull the Salary column out as a plain NumPy array
    for _ in range(reps):                          # repeat the sampling process 'reps' times (e.g. 1000)
        sample = rng.choice(values, size=n, replace=False)  # draw n values at random without replacement
        means.append(sample.mean())                # record the mean of this particular sample
    return np.array(means)                         # convert the list of 1000 sample means into a NumPy array

means_n5 = simulate_sampling_dist(df, 5, reps=1000, seed=1)                       # simulate 1000 sample means, each from a sample of size 5
means_n30 = simulate_sampling_dist(df, min(30, len(df)), reps=1000, seed=2)       # simulate 1000 sample means from samples of size 30 (capped at population size)

print(f"n=5  -> mean of sample means = {means_n5.mean():,.2f}, std = {means_n5.std():,.2f}")
print(f"n=30 -> mean of sample means = {means_n30.mean():,.2f}, std = {means_n30.std():,.2f}")
print(f"\nPopulation std / sqrt(5)  = {population_std/np.sqrt(5):,.2f}")     # CLT prediction for the standard error at n=5
print(f"Population std / sqrt(30) = {population_std/np.sqrt(30):,.2f}")      # CLT prediction for the standard error at n=30

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))          # 1 row, 2 columns of subplots, side by side

axes[0].hist(means_n5, bins=25, color='cornflowerblue', edgecolor='black')   # histogram of the 1000 sample means (n=5 each)
axes[0].set_title('Sampling Distribution of Mean Salary (n=5)')
axes[0].set_xlabel('Sample Mean'); axes[0].set_ylabel('Frequency')

axes[1].hist(means_n30, bins=25, color='indianred', edgecolor='black')      # histogram of the 1000 sample means (n=30 each)
axes[1].set_title('Sampling Distribution of Mean Salary (n=30)')
axes[1].set_xlabel('Sample Mean'); axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

**7.4 Observations:**
- **Shape:** at n = 5 the distribution of sample means is more spread out and somewhat rougher;
  at n = 30 (here effectively resampling the whole population without replacement, so variance
  is very small) the distribution becomes much tighter and more symmetric/bell-shaped —
  consistent with the CLT even though the underlying population itself is not huge.
- **Spread vs. population SD:** the standard deviation of the simulated sample means shrinks as
  n grows, tracking the CLT prediction σ/√n closely (see printed comparison above) — quadrupling
  n roughly halves the standard error, matching the √n relationship, not a linear one.
- **CLT in plain words:** even though individual engineers' salaries are not perfectly normally
  distributed, the *average* salary computed from repeated random samples becomes more tightly
  clustered around the true population mean, and its distribution looks increasingly bell-shaped,
  as the sample size n increases. This is why larger HR salary surveys give more reliable and
  precise estimates of the true average salary than small ones.

## Task 8: Confidence Intervals (Parametric)

In [ ]:
def t_confidence_interval(data, confidence=0.95):
    n = len(data)                                   # sample size
    mean = np.mean(data)                            # sample mean (our point estimate, the center of the interval)
    sem = stats.sem(data)                           # standard error of the mean = sample_std / sqrt(n)
    t_crit = stats.t.ppf((1+confidence)/2, df=n-1)  # critical t-value for a 2-sided 95% interval, with n-1 degrees of freedom
    margin = t_crit * sem                           # margin of error = t_critical * standard error
    return mean, mean - margin, mean + margin, margin   # return mean, lower bound, upper bound, and margin

mean_full, low_full, high_full, margin_full = t_confidence_interval(df['Salary'])   # 95% CI using ALL 30 records
print(f"Full data (n={len(df)}): mean = {mean_full:,.2f}")
print(f"95% CI: ({low_full:,.2f}, {high_full:,.2f})  -> width = {high_full-low_full:,.2f}")

**8.2 Interpretation for management:** "We are 95% confident that the true average salary
of engineers in this company lies between **the lower and upper bound printed above**." (In
repeated sampling, 95% of such intervals constructed this way would contain the true population
mean.)

In [ ]:
sample15 = df.sample(n=15, random_state=7)                                # draw one random sample of 15 engineers
mean_15, low_15, high_15, margin_15 = t_confidence_interval(sample15['Salary'])  # 95% CI computed from just this smaller sample
print(f"Sample (n=15): mean = {mean_15:,.2f}")
print(f"95% CI: ({low_15:,.2f}, {high_15:,.2f})  -> width = {high_15-low_15:,.2f}")

print(f"\nCI width, full data (n={len(df)}) = {high_full-low_full:,.2f}")   # recall the CI width from the full-data case above
print(f"CI width, sample (n=15)          = {high_15-low_15:,.2f}")          # compare it directly to the smaller sample's CI width

**Comparison:** the confidence interval from the smaller sample (n = 15) is **wider** than
the one from the full dataset. This is because the margin of error is proportional to
s/√n × t-critical — a smaller n means a larger standard error (less information about the
population) and a larger t-critical value (heavier tails for small df), both of which widen
the interval. Larger samples give more precise (narrower) estimates of the population mean.

## Task 9: Hypothesis Testing – Parametric Tests

### 9.1 One-Sample t-Test: is mean salary = 80,000?

H0: μ = 80,000  vs.  H1: μ ≠ 80,000 (two-sided), α = 0.05

In [ ]:
t_stat, p_value = stats.ttest_1samp(df['Salary'], popmean=80000)   # one-sample t-test: compares the sample mean to the hypothesized value 80,000
print(f"Sample mean salary = {df['Salary'].mean():,.2f}")
print(f"t-statistic = {t_stat:.4f}")           # how many standard errors the sample mean is away from 80,000
print(f"p-value     = {p_value:.6f}")          # probability of seeing a difference this large (or larger) if H0 were actually true
alpha = 0.05                                   # our chosen significance level (5% chance of wrongly rejecting a true H0)
decision = "Reject H0" if p_value < alpha else "Fail to reject H0"   # standard decision rule: reject H0 if p-value is below alpha
print(f"Decision at alpha=0.05: {decision}")

**Interpretation:** Given the p-value relative to α = 0.05, we (reject / fail to reject)
H0 as printed above. In business terms — if H0 is rejected, there is statistically significant
evidence that the company's average engineer salary differs from the claimed industry figure of
80,000, and management should investigate whether pay is over- or under-shooting the industry
benchmark. If H0 is not rejected, the data is consistent with the company's average salary
being in line with 80,000 (though this does not *prove* equality, only that we lack sufficient
evidence of a difference).

### 9.2 Two-Sample t-Test: Junior vs. Senior engineers

H0: mean salary (Junior) = mean salary (Senior)  vs.  H1: means differ

In [ ]:
junior = df[df['YearsExperience'] <= 3]['Salary']    # Salary values for engineers with <= 3 years experience (Junior group)
senior = df[df['YearsExperience'] >= 7]['Salary']    # Salary values for engineers with >= 7 years experience (Senior group)

print(f"Junior: n={len(junior)}, mean={junior.mean():,.2f}, std={junior.std():,.2f}")
print(f"Senior: n={len(senior)}, mean={senior.mean():,.2f}, std={senior.std():,.2f}")

# Check equal variance assumption with Levene's test
levene_stat, levene_p = stats.levene(junior, senior)          # Levene's test: checks whether the two groups have statistically similar variances
print(f"\nLevene's test for equal variances: stat={levene_stat:.4f}, p={levene_p:.4f}")
equal_var = levene_p > 0.05                                   # if Levene's p-value > 0.05, we don't have evidence variances differ -> treat as equal
print(f"Assume equal variances: {equal_var}")

t_stat2, p_value2 = stats.ttest_ind(junior, senior, equal_var=equal_var)   # independent two-sample t-test; uses pooled or Welch formula depending on equal_var
print(f"\nTwo-sample t-test ({'pooled' if equal_var else 'Welch'}):")
print(f"t-statistic = {t_stat2:.4f}")
print(f"p-value     = {p_value2:.6f}")
decision2 = "Reject H0" if p_value2 < 0.05 else "Fail to reject H0"
print(f"Decision at alpha=0.05: {decision2}")

**Interpretation:** Levene's test checks whether the two groups' variances are similar
enough to use the standard pooled-variance t-test (if p > 0.05, variances are treated as
equal) or whether Welch's t-test (unequal variance) is more appropriate. The resulting p-value
for the group-mean comparison, given the very large gap in mean salary between Junior and
Senior engineers, is expected to be highly significant — i.e., **experience level has a
statistically significant effect on salary** in this dataset, reinforcing the strong
correlation found in Task 4.

## Task 10: Non-Parametric Hypothesis Testing

### 10.1 Mann–Whitney U Test (Junior vs. Senior)

In [ ]:
u_stat, u_p = stats.mannwhitneyu(junior, senior, alternative='two-sided')  # rank-based test comparing the two groups' distributions (no normality assumption)
print(f"Mann-Whitney U statistic = {u_stat:.4f}")
print(f"p-value = {u_p:.6f}")
decision_mw = "Reject H0" if u_p < 0.05 else "Fail to reject H0"
print(f"Decision at alpha=0.05: {decision_mw}")

**Comparison with the t-test:** the Mann–Whitney U test (which compares distributions/ranks
rather than means) reaches the same qualitative conclusion as the two-sample t-test in Task 9.2 —
both indicate a significant difference between Junior and Senior salaries. We might prefer a
non-parametric test when: (a) the sample size is small (as here, n=30) and normality is uncertain,
(b) there are outliers that would distort a mean-based test, or (c) the data is ordinal or
clearly non-normal — Mann–Whitney is more robust in these situations since it only relies on
rank ordering, not distributional assumptions.

### 10.2 One-Sample Wilcoxon Signed-Rank Test

H0: median salary = 80,000

In [ ]:
w_stat, w_p = stats.wilcoxon(df['Salary'] - 80000)   # Wilcoxon signed-rank test: tests whether the median of (Salary - 80000) differs from 0, i.e. whether median Salary differs from 80,000
print(f"Sample median salary = {df['Salary'].median():,.2f}")
print(f"Wilcoxon signed-rank statistic = {w_stat:.4f}")
print(f"p-value = {w_p:.6f}")
decision_w = "Reject H0" if w_p < 0.05 else "Fail to reject H0"
print(f"Decision at alpha=0.05: {decision_w}")

**Interpretation:** Based on the p-value above, we (reject / fail to reject) the
hypothesis that the median salary equals 80,000. Note this test assumes the differences
(Salary − 80,000) are roughly symmetric, which is a lighter assumption than full normality —
worth mentioning as a limitation for a dataset of only 30 observations.

## Task 11: One-Way and Two-Way ANOVA

### 11.1 One-Way ANOVA — Experience Level → Salary

In [ ]:
def exp_level(y):
    if y < 3: return 'Level1_Junior'      # experience below 3 years -> Junior
    elif y < 7: return 'Level2_Mid'       # experience between 3 and 7 years -> Mid
    else: return 'Level3_Senior'          # experience 7 years or more -> Senior

df['ExpLevel'] = df['YearsExperience'].apply(exp_level)   # apply the function row-by-row to create a categorical ExpLevel column
print(df['ExpLevel'].value_counts())                      # how many engineers fall in each experience level group

groups = [df[df['ExpLevel']==lvl]['Salary'] for lvl in df['ExpLevel'].unique()]  # build a list of Salary Series, one per experience-level group
f_stat, p_val_anova = stats.f_oneway(*groups)             # one-way ANOVA: tests whether the group means are all equal; *groups unpacks the list as separate arguments
print(f"\nOne-Way ANOVA: F-statistic = {f_stat:.4f}, p-value = {p_val_anova:.6f}")
decision_anova = "Reject H0" if p_val_anova < 0.05 else "Fail to reject H0"
print(f"Decision at alpha=0.05: {decision_anova}")

**Interpretation:** The F-statistic and p-value indicate whether mean salary differs
significantly across the three experience levels. Given the strong experience–salary
relationship already established, we expect to **reject H0**, concluding that experience
level has a statistically significant effect on salary.

### 11.2 Two-Way ANOVA — Experience Level × Department (synthetic)

In [ ]:
rng = np.random.default_rng(123)                                          # seeded random generator for reproducibility
df['Department'] = rng.choice(['Design', 'Testing', 'Management'], size=len(df))  # randomly assign each engineer to one of 3 synthetic departments (no real relationship to salary)

model = ols('Salary ~ C(ExpLevel) + C(Department) + C(ExpLevel):C(Department)', data=df).fit()
# ^ fits a linear model with: main effect of ExpLevel, main effect of Department, and their interaction.
#   C(...) tells statsmodels to treat the variable as categorical, not numeric.
anova_table = sm.stats.anova_lm(model, typ=2)   # convert the fitted model into a Type II ANOVA table (F-stats and p-values per factor)
anova_table                                     # display the table

**Interpretation:**
- **Main effect of Experience Level:** expected to be statistically significant (low p-value),
  consistent with Task 11.1 — real signal in the data.
- **Main effect of Department:** since `Department` was randomly (synthetically) assigned with
  no real relationship to salary, its p-value is expected to be **not significant**.
- **Interaction (Experience × Department):** likewise expected to be **not significant**, since
  department was assigned independently of salary.

This is a useful illustration: ANOVA correctly distinguishes the *real* driver of salary
(experience) from a *random, non-informative* factor (department), which is exactly the kind
of check a data analyst should run before attributing pay differences to a given factor.

## Task 12: Engineering Application & Interpretation Report

**To:** HR and Engineering Management
**From:** Data Analytics Team
**Re:** Statistical Analysis of Engineer Salary Data

---

**1. Alignment with industry/target average salary**
Using a one-sample t-test against the claimed industry average of ₹80,000, the analysis
(Task 9.1) found the company's average salary to be [state result from the printed t-test:
either "not statistically distinguishable from ₹80,000" or "statistically different from
₹80,000, at ₹X"]. The 95% confidence interval for the true mean salary (Task 8) provides a
plausible range for the actual company-wide average, which management can compare directly
against external benchmarking data. Practically, if the interval sits above or below 80,000,
this should prompt a discussion on whether the company is over- or under-paying relative to
the stated industry figure.

**2. Evidence for salary differences based on experience (and department)**
Both the parametric (two-sample t-test, one-way ANOVA) and non-parametric (Mann–Whitney U)
tests consistently show a **statistically significant salary difference between Junior and
Senior engineers**, and across the three experience levels overall (Task 9.2, 10.1, 11.1).
The correlation between experience and salary (Task 4) is strongly positive, confirming that
experience is the dominant, legitimate driver of pay in this dataset. The synthetic Department
variable, by contrast, showed **no significant main effect or interaction effect** on salary
(Task 11.2) — as expected, since it carries no real information, but this exercise demonstrates
that ANOVA can help separate genuine drivers of pay from spurious or irrelevant ones. In a
real dataset, if an actual department variable were found similarly insignificant, that would
be reassuring evidence against unwarranted departmental pay disparities; if significant, it
would warrant further investigation into whether the difference is justified by role
complexity/scope or reflects unfair disparity.

**3. Practical recommendations**
- **Adjusting junior vs. senior salary bands:** Given the statistically robust and large gap
  between Junior and Senior pay, management should formalize clear, transparent salary bands
  tied to experience milestones, ensuring the progression is consistent and defensible (and
  periodically benchmarked against the external industry average from Task 9.1).
- **Performance-based incentives:** Since salary here is explained almost entirely by
  experience (Task 4's high correlation), consider layering performance-based components
  (bonuses, merit increases) so that pay also reflects contribution/output, not tenure alone —
  this can improve motivation without disrupting the fair experience-based base structure.
- **Need for further data collection:** This analysis relies on only two variables
  (experience and salary) from 30 records. To make more defensible compensation decisions,
  HR should collect additional fields — education level, specific role/title, project outcomes,
  performance ratings, and actual department — and re-run this analysis (regression/ANOVA) to
  identify which factors *legitimately* explain pay, and to check for any unexplained
  disparities that may need correcting.

---
*Note: bracketed placeholders above should be filled in with the specific numeric results
obtained when this notebook is executed, since exact t-statistics/p-values/CI bounds depend
on the random seeds and the actual data.*